# Проверка слабых кандидатов при перекрытии

Подключите конкурсные файлы и приватный clevr_rear. Выберите GPU T4 ×2. Исходная модель для сравнения — count_hidden. Каждый GPU обучает независимый seed.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("rear_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_rear.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_rear.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/rear_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

from scipy.special import logit

## Извлечение кандидатов, обучение и валидация

In [ ]:
env.update(FRAGMENT_INPUT=str(package),OMP_NUM_THREADS="2",OPENBLAS_NUM_THREADS="2")
seeds=(421,853)
root_out=ROOT/"rear_result"
def run(gpu,action,split,shard,seed):
    stage=f"{action}_{split}_{shard}_{seed}"
    done=ROOT/f"{stage}.done"
    if done.exists():return
    cmd=[sys.executable,"-m","scripts.research.rear_candidate",action,"--split",split,
         "--shard",str(shard),"--seed",str(seed)]
    with (RESULT/f"{stage}.log").open("a") as log, subprocess.Popen(cmd,cwd=ROOT,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)},stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            log.write(line)
            print(f"[T4 {gpu}] {line}",end="",flush=True)
        if proc.wait():
            raise RuntimeError(f"Не завершился этап {stage}. Повторите эту ячейку.")
    done.touch()

def both(left,right):
    with ThreadPoolExecutor(2) as pool:
        futures=[pool.submit(run,i,*args) for i,args in enumerate((left,right))]
        for future in futures:future.result()

def prepare(split):
    both(*[("prepare",split,i,421) for i in (0,1)])
def predict(split):
    prepare(split)
    for seed in seeds:
        both(*[("predict",split,i,seed) for i in (0,1)])

prepare("train")
both(*[("fit","train",0,s) for s in seeds])
predict("val")

## AUC, исправленные пары и кандидаты на сложных сценах

In [ ]:
frame=pd.read_csv(DATA/"val.csv")
CORE=("counting","comparison","spatial_counting")
def deltas(split,df):
    arms=[]
    accepted=[]
    for seed in seeds:
        chunks=[]
        for shard in (0,1):
            with np.load(root_out/f"{split}_{seed}_{shard}.npz") as a:
                chunks.append(pd.DataFrame(dict(delta=a["delta"],selected=a["selected"]),index=a["ids"]))
        correction=pd.concat(chunks)
        assert correction.index.is_unique and correction.index.isin(df.id).all()
        assert np.isfinite(correction.delta).all()
        change=correction.delta.reindex(df.id,fill_value=0.).to_numpy()
        arms.append(np.where(df.category.isin(CORE),change,0.))
        accepted.append(correction.selected.reindex(df.id,fill_value=-1).to_numpy())
    agree=(accepted[0]>=0)&(accepted[0]==accepted[1])
    return arms,np.where(agree,np.mean(arms,axis=0),0.)

with np.load(package/"anchor_val.npz") as a:
    assert np.array_equal(a["ids"],frame.id)
    base=a["score"]
    confirmed=a["confirmed"]
assert abs(roc_auc_score(frame.label,base)-.9998736666666667)<1e-10
arms,revision=deltas("val",frame)
score=base+revision
y=frame.label.to_numpy()
scenes=pd.factorize(frame.image)[0]
n=scenes.max()+1
resamples=np.random.default_rng(705).multinomial(n,np.full(n,1/n),size=1000)[:,scenes]
def measure(x):
    roc=roc_auc_score(y,x)
    errs=((x<=0)&(y==1)).sum()
    return dict(auc=float(roc),wrong_or_tied=int(errs),inversions=float(round((1-roc)*9_000_000,2)))
metrics={k:measure(x) for k,x in [
    ("r50_count",confirmed),("count_hidden",base),("seed421",base+arms[0]),
    ("seed853",base+arms[1]),("rear",score)]}
display(pd.DataFrame(metrics).T)
diff=[roc_auc_score(y,score,sample_weight=w)-roc_auc_score(y,base,sample_weight=w) for w in resamples]
ci=np.quantile(diff,[.025,.975])
joined=frame.assign(base=base,rear=score,change=score-base)
joined.to_csv(RESULT/"val_rows.csv",index=False)
changed=joined[joined.label.eq(1)&((joined.base<=0)|(joined.rear<=0))]
display(changed[["image","caption","base","rear","change"]].sort_values("base"))
records=[]
for path in sorted(root_out.glob("candidates_*.json")):
    for row in json.loads(path.read_text(encoding="utf8")):
        records.append(dict(run=path.stem,**row))
cards=pd.DataFrame(records)
weak_counts={}
if len(cards):
    display(cards[cards.image.isin(["58281.jpg","78612.jpg","26268.jpg"])])
    cards["seed"]=cards.run.str.split("_").str[1]
    for seed,group in cards.groupby("seed"):
        weak_counts[seed]={}
        for label in (0,1):
            known=group[group.weak_target.eq(label)]
            weak_counts[seed][str(label)]=dict(total=len(known),promoted=int(known.used.sum()))
    print("Слабая разметка по точным подписям val",weak_counts)
# A single candidate may independently change a twin sign or only its rank.
fixed=int(((y==1)&(base<=0)&(score>0)).sum())
new_errors=int(((y==1)&(base>0)&(score<=0)).sum())
passed=bool(metrics["rear"]["auc"]>metrics["count_hidden"]["auc"] and new_errors<=fixed)
(RESULT/"metrics.json").write_text(json.dumps(dict(metrics=metrics,ci=ci.tolist(),fixed=fixed,
    new_errors=new_errors,caption_constraints=weak_counts,test=passed),indent=2),encoding="utf8")
print("95% интервал изменения AUC",ci,"Исправлено",fixed,"Новых ошибок",new_errors,"Инференс test",passed)

## Архив и условный инференс на test

In [ ]:
if passed:
    predict("test")
    test=pd.read_csv(DATA/"test.csv")
    with np.load(package/"anchor_test.npz") as a:
        assert np.array_equal(a["ids"],test.id)
        probability=a["probability"]
    _,change=deltas("test",test)
    update=change!=0
    probability[update]=expit(logit(probability[update].clip(1e-15,1-1e-15))+change[update])
    sample=pd.read_csv(DATA/"sample_submission.csv")
    assert sample.id.is_unique and test.id.is_unique and set(sample.id)==set(test.id)
    sample["label"]=pd.Series(probability,index=test.id).loc[sample.id].to_numpy()
    path=WORK/"count_rear.csv"
    sample.to_csv(path,index=False,float_format="%.17g")
    assert sample.equals(pd.read_csv(path,float_precision="round_trip")) and sample.label.between(0,1).all()
with zipfile.ZipFile(WORK/"rear_results.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for folder in (RESULT,root_out):
        for path in folder.iterdir():
            if path.is_file():
                z.write(path,path.relative_to(ROOT))
os.chdir(WORK)
display(FileLink("rear_results.zip"))
if passed:
    display(FileLink("count_rear.csv"))